# Unconditional SE(3) Diffusion Training

Trains an **unconditional** SE(3) score network on a single protein MD
trajectory following the SinFusion single-trajectory protocol.

**What you get:** A model that generates new protein backbone conformations
from noise, sampling from the learned equilibrium distribution.

| Step | Description |
|------|-------------|
| 1 | Mount Google Drive (persistent storage) |
| 2 | Clone repo & init submodules |
| 3 | Configure protein & training |
| 4 | Download & preprocess trajectory |
| 5 | Train |
| 6 | Generate samples |

## Step 1: Environment Setup

In [ ]:
# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

In [ ]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/protein_data/data', exist_ok=True)
    print('Drive mounted')

In [ ]:
# Install dependencies

# Core (required for training)
!pip install -q pytorch-lightning omegaconf biopython dm-tree ml-collections wandb

# Evaluation: mdtraj, deeptime (TICA), statsmodels (autocorrelation)
!pip install -q 'numpy<2' mdtraj statsmodels deeptime

# Optional: pyemma (uses MDGen's full analysis script; falls back to deeptime if unavailable)
!pip install -q pyemma 2>/dev/null || echo 'pyemma not installed — will use deeptime fallback for TICA'
# MDGen baseline + optional
!pip install -q huggingface-hub py3Dmol 2>/dev/null || true

# Verify
!python -c 'import mdtraj; print(f"mdtraj {mdtraj.__version__} OK")'
!python -c 'import deeptime; print(f"deeptime {deeptime.__version__} OK")'
!python -c 'import numpy; print(f"numpy {numpy.__version__}")'

print('Dependencies installed')


## Step 2: Clone Repository & Init Submodules

In [ ]:
import os, subprocess, shutil

REPO_URL = 'https://github.com/JiwonJJeong/winter-gen-pproject.git'

if IN_COLAB:
    os.chdir('/content')
    # Fresh clone each time to avoid dirty-state conflicts
    if os.path.exists('winter-gen-pproject'):
        shutil.rmtree('winter-gen-pproject')
    subprocess.run(['git', 'clone', '--recurse-submodules', REPO_URL], check=True)
    os.chdir('winter-gen-pproject')
    # Symlink data/ to Drive for persistence
    if not os.path.islink('data'):
        os.symlink('/content/drive/MyDrive/protein_data/data', 'data')
        print('data/ -> /content/drive/MyDrive/protein_data/data')
    print('Code ready')
else:
    assert os.path.isdir('gen_model'), 'Run from repo root'
    print(f'Local repo: {os.path.abspath(".")}')


## Step 3: Configuration

Edit the variables below for your protein and training preferences.

In [ ]:
# ---- Protein (single-trajectory, SinFusion-style) ----
PROTEIN   = '4o66_C'   # Protein name (as in atlas.csv)
REPLICA   = '1'        # Which replica (1, 2, or 3)

# ---- Generalization test (ill-posed but informative) ----
EVAL_PROTEIN = '6ono_C'  # 76-residue protein from MDGen/STAR-MD test set (never in any training split)

# ---- Training ----
MAX_STEPS  = 50_000    # 50k for Colab; 200k for full training
BATCH_SIZE = 1         # SinFusion default for single-trajectory
LR         = 1e-4
GRAD_CLIP  = 1.0
EMA_DECAY  = 0.999

# ---- Architecture ----
NUM_BLOCKS = 3     # IPA blocks (3 for single-trajectory; paper default: 8)
CROP_RATIO = 0.7   # Residue crop ratio (SinFusion anti-overfitting: 0.7; MDGen default: 0.95)

# ---- Weights & Biases ----
USE_WANDB     = True   # Set False to disable W&B logging
WANDB_PROJECT = 'winter-gen'

# ---- Paths (Drive on Colab, local otherwise) ----
DRIVE_BASE = '/content/drive/MyDrive/protein_data' if IN_COLAB else '.'
DATA_DIR   = f'{DRIVE_BASE}/data'
SAVE_DIR   = f'{DRIVE_BASE}/checkpoints/unconditional/{PROTEIN}'
OUT_DIR    = f'{DRIVE_BASE}/outputs/unconditional/{PROTEIN}'

print(f'Train protein : {PROTEIN}_R{REPLICA}')
print(f'Eval protein  : {EVAL_PROTEIN}')
print(f'Steps         : {MAX_STEPS:,}')
print(f'Blocks        : {NUM_BLOCKS}')
print(f'Save to       : {SAVE_DIR}')
print(f'Outputs to    : {OUT_DIR}')


## Step 4: Download & Prepare Data

In [ ]:
import os

if IN_COLAB and not os.path.islink('data'):
    raise RuntimeError('data/ not linked to Drive. Run Step 1 first.')

# Download training protein
!PYTHONPATH=. python scripts/download_and_prep.py {PROTEIN} --data_dir {DATA_DIR} --out_dir {DATA_DIR} --suffix _latent

# Download evaluation protein (for generalization test)
!PYTHONPATH=. python scripts/download_and_prep.py {EVAL_PROTEIN} --data_dir {DATA_DIR} --out_dir {DATA_DIR} --suffix _latent

for prot in [PROTEIN, EVAL_PROTEIN]:
    traj_path = f'{DATA_DIR}/{prot}/{prot}_R1_latent.npy'
    if os.path.exists(traj_path):
        import numpy as np
        arr = np.load(traj_path)
        print(f'  {prot}: {traj_path}  shape={arr.shape}')
    else:
        print(f'  ERROR: {traj_path} not found')


## Step 5: Train (Stage 1 — Unconditional)

Learns SE(3) backbone geometry from single frames. The checkpoint produced here is the warm-start for Stage 2 (conditional notebook).

Runs `gen_model/train_unconditional.py` with:
EMA, gradient clipping, cosine LR, mixed precision.

In [ ]:
# ---- Weights & Biases setup ----
# Log in once per Colab session. Your API key is at https://wandb.ai/authorize
import wandb
wandb.login()  # prompts for API key if not already authenticated
print('W&B ready')


In [ ]:
wandb_flag = '--wandb' if USE_WANDB else ''

!PYTHONPATH=. python gen_model/train_unconditional.py \
    --protein {PROTEIN} \
    --replica {REPLICA} \
    --data_dir {DATA_DIR} \
    --batch_size {BATCH_SIZE} \
    --max_steps {MAX_STEPS} \
    --lr {LR} \
    --grad_clip {GRAD_CLIP} \
    --ema_decay {EMA_DECAY} \
    --save_dir {SAVE_DIR} \
    --num_blocks {NUM_BLOCKS} \
    --crop_ratio {CROP_RATIO} \
    --num_workers 2 \
    {wandb_flag} \
    --wandb_project {WANDB_PROJECT}

In [ ]:
import os
stage1_ckpt = os.path.join(SAVE_DIR, 'last.ckpt')
print(f'Stage 1 checkpoint: {stage1_ckpt}')
print()
print('Use this path as UNCOND_CKPT in the conditional notebook (Stage 2).')


## Step 6: Generate Samples (Training Protein)

In [ ]:
import glob

ckpts = sorted(glob.glob(f'{SAVE_DIR}/uncond-*.ckpt'))
best_ckpt = ckpts[-1] if ckpts else f'{SAVE_DIR}/last.ckpt'
print(f'Checkpoint: {best_ckpt}')

!PYTHONPATH=. python gen_model/inference_unconditional.py \
    --checkpoint {best_ckpt} \
    --npy_path {DATA_DIR}/{PROTEIN}/{PROTEIN}_R{REPLICA}_latent.npy \
    --atlas_csv gen_model/splits/atlas.csv \
    --protein_name {PROTEIN} \
    --num_samples 100 \
    --num_steps 200 \
    --out_dir {OUT_DIR}/../unconditional/{PROTEIN}


## Step 7: Evaluate (Training Protein)

Compare generated samples against reference MD (all 3 replicas).

In [ ]:
!PYTHONPATH=. python gen_model/evaluate.py \
    --ref_npy {DATA_DIR}/{PROTEIN}/{PROTEIN}_R1_latent.npy \
              {DATA_DIR}/{PROTEIN}/{PROTEIN}_R2_latent.npy \
              {DATA_DIR}/{PROTEIN}/{PROTEIN}_R3_latent.npy \
    --gen_traj {DRIVE_BASE}/outputs/unconditional/{PROTEIN}/ca_samples.npy \
    --protein {PROTEIN} --mode unconditional --ref_mode all \
    --out_dir {OUT_DIR}/../eval/unconditional/{PROTEIN}

# Display the evaluation plots (generated by MDGen analysis)
eval_pdf = f'{DRIVE_BASE}/outputs/eval/unconditional/' + PROTEIN + '/gen/' + PROTEIN + '.pdf'
if os.path.exists(eval_pdf):
    from IPython.display import display, IFrame
    display(IFrame(eval_pdf, width=800, height=600))
else:
    print(f'No evaluation PDF found at {eval_pdf}')


## Step 8: Visualize (Training Protein)

In [ ]:
from gen_model.viz import _viz_ca
import numpy as np, os

ca_path = f'{DRIVE_BASE}/outputs/unconditional/{PROTEIN}/ca_samples.npy'
if os.path.exists(ca_path):
    samples = np.load(ca_path)  # [S, N, 3]
    _viz_ca(samples, f'{PROTEIN} (training protein) — {len(samples)} samples')
else:
    print('No samples found. Run inference first.')


## Step 9: Generalization Test

Generate and evaluate on `EVAL_PROTEIN` — a held-out protein from the
MDGen/STAR-MD test set that was never in any training split.
This quantifies how much the single-trajectory model overfits vs. generalizes.

In [ ]:
print(f'Generalization test: {PROTEIN} (train) -> {EVAL_PROTEIN} (held-out)')
print(f'  {EVAL_PROTEIN} is from the MDGen/STAR-MD test set')

# Generate on held-out protein
!PYTHONPATH=. python gen_model/inference_unconditional.py \
    --checkpoint {best_ckpt} \
    --npy_path {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R1_latent.npy \
    --atlas_csv gen_model/splits/atlas.csv \
    --protein_name {EVAL_PROTEIN} \
    --num_samples 100 \
    --num_steps 200 \
    --out_dir {OUT_DIR}/../unconditional/{EVAL_PROTEIN}

# Evaluate against all 3 replicas
!PYTHONPATH=. python gen_model/evaluate.py \
    --ref_npy {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R1_latent.npy \
              {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R2_latent.npy \
              {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R3_latent.npy \
    --gen_traj {DRIVE_BASE}/outputs/unconditional/{EVAL_PROTEIN}/ca_samples.npy \
    --protein {EVAL_PROTEIN} --mode unconditional --ref_mode all \
    --out_dir {OUT_DIR}/../eval/unconditional/{EVAL_PROTEIN}

# Display evaluation PDFs
import glob as _glob
from IPython.display import display, IFrame
for _pdf in _glob.glob(f'{DRIVE_BASE}/outputs/**/eval*/**/*.pdf', recursive=True)[-3:]:
    if _pdf not in globals().get('_shown_pdfs', set()):
        print(f'\n{_pdf}:')
        display(IFrame(_pdf, width=800, height=600))
        _shown_pdfs = globals().get('_shown_pdfs', set()) | {_pdf}


In [ ]:
# Visualize generalization samples
import numpy as np
import matplotlib.pyplot as plt

ca_path = f'{DRIVE_BASE}/outputs/unconditional/{EVAL_PROTEIN}/ca_samples.npy'
if os.path.exists(ca_path):
    samples = np.load(ca_path)
    print(f'Generated: {samples.shape}')
    fig, axes = plt.subplots(1, min(5, len(samples)), figsize=(15, 3))
    if len(samples) == 1: axes = [axes]
    for i, ax in enumerate(axes):
        ca = samples[i]
        ax.plot(ca[:, 0], ca[:, 1], '-', lw=0.5)
        ax.set_title(f'Sample {i+1}')
        ax.set_aspect('equal'); ax.axis('off')
    plt.suptitle(f'{EVAL_PROTEIN} (held-out protein) — Generated CA traces')
    plt.tight_layout(); plt.show()
else:
    print('No samples found.')
